In [1]:
# Hypothesis evaluations, p-value calculations, A/B logic

# Import & Connect to Relational Warehouse
import os
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats  # Exact library used in Slide 13 & 22

BASE_DIR = os.path.dirname(os.path.abspath(""))
DB_PATH = os.path.join(BASE_DIR, "data-package", "database", "afilearn_commerce_master.db")

# Read our high-integrity, engineered production tables
conn = sqlite3.connect(DB_PATH)
df_orders = pd.read_sql_query("SELECT order_id, delivery_delay_days, is_late_delivery FROM prod_orders_cleaned;", conn)
df_reviews = pd.read_sql_query("SELECT order_id, review_score FROM prod_reviews_cleaned;", conn)
conn.close()

# Merge tables on order_id
df_analysis = pd.merge(df_orders, df_reviews, on="order_id", how="inner")
df_analysis['review_score'] = pd.to_numeric(df_analysis['review_score'])
df_analysis['delivery_delay_days'] = pd.to_numeric(df_analysis['delivery_delay_days'])

print(f"✔ Dataset synchronized. Ready to evaluate {len(df_analysis):,} transacted records under uncertainty.")

✔ Dataset synchronized. Ready to evaluate 95,607 transacted records under uncertainty.


In [2]:
# Welch's Two-Sample t-Test Execution (Unequal Variances)
group_on_time = df_analysis[df_analysis['is_late_delivery'] == 0]['review_score']
group_late = df_analysis[df_analysis['is_late_delivery'] == 1]['review_score']

# Run Welch's t-test using SciPy
t_stat, p_val = stats.ttest_ind(group_late, group_on_time, equal_var=False, alternative="less")

# Calculate metrics for the Report
mean_ontime = group_on_time.mean()
mean_late = group_late.mean()
effect_size = mean_ontime - mean_late

# Calculate confidence interval for the difference (Slide 33)
n_ontime, n_late = len(group_on_time), len(group_late)
se_diff = np.sqrt(group_on_time.var()/n_ontime + group_late.var()/n_late)
ci_diff = (effect_size - 1.96 * se_diff, effect_size + 1.96 * se_diff)

print("=== WELCH'S TWO-SAMPLE T-TEST RESULTS ===")
print(f"On-Time Mean Score: {mean_ontime:.4f} (n = {n_ontime:,})")
print(f"Late Mean Score: {mean_late:.4f} (n = {n_late:,})")
print(f"Observed Drop (Effect Size): -{effect_size:.4f} score points")
print(f"95% CI for Score Drop: [{ci_diff[0]:.4f}, {ci_diff[1]:.4f}]")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_val}")

=== WELCH'S TWO-SAMPLE T-TEST RESULTS ===
On-Time Mean Score: 4.2922 (n = 89,245)
Late Mean Score: 2.2700 (n = 6,362)
Observed Drop (Effect Size): -2.0222 score points
95% CI for Score Drop: [1.9828, 2.0615]
t-statistic: -100.7804
p-value: 0.0


# Phase 4: Statistical Hypothesis Testing

Following the structured **Five-Step testing workflow**, we evaluate the impact of logistics delays on customer satisfaction under operational uncertainty.

---

### Step 1: State the Decision Question
> **"Should AfiLearn invest capital to build, train, and deploy an early-warning predictive machine learning system to intercept and remediate delayed shipments?"**

---

### Step 2: Formulate Hypotheses
To assess whether the observed drop in satisfaction is statistically reliable, we set our default assumption against our claim:

* **Null Hypothesis ($H_0$):** $\mu_{\text{late}} \ge \mu_{\text{on-time}}$  
    *(The average customer review score for delayed deliveries is equal to or greater than on-time/early deliveries; delays have no negative impact on satisfaction).*
* **Alternative Hypothesis ($H_1$):** $\mu_{\text{late}} < \mu_{\text{on-time}}$  
    *(The average customer review score for delayed deliveries is significantly lower than for on-time/early deliveries).*

---

### Step 3: Test Specifications & Decision Rules (Slide 17, 18, 24)
* **Significance Level ($\alpha$):** $0.05$ (Standard business confidence threshold).
* **Test Method:** **Two-Sample Welch's t-Test** (Selected because our groups have unequal sample sizes ($n_1 \ne n_2$) and unequal variances ($\sigma^1 \ne \sigma^2$)).
* **Decision Rule:** Reject $H_0$ if the calculated $p\text{-value} \le \alpha$ (Slide 18).

---

### 💻 Step 4: Empirical Test Computations & Results (Slide 17)

We evaluated a total sample size of **$95,607$ transacted records** to maximize statistical power and eliminate Type II errors (Slide 35, 45).

| Parameter | Metric Value | Analysis & Meaning |
| :--- | :--- | :--- |
| **On-Time Sample ($n_1$)** | **$89,245$** | Represents the operational baseline of early or on-time fulfillment. |
| **Late Sample ($n_2$)** | **$6,362$** | Represents transacted orders that crossed the estimated delivery date. |
| **On-Time Mean Score ($\bar{x}_1$)** | **$4.2922$ / $5.0$** | High baseline satisfaction for timely deliveries. |
| **Late Mean Score ($\bar{x}_2$)** | **$2.2700$ / $5.0$** | Drastic drop-off in user satisfaction once a shipment is late. |
| **Observed Drop (Effect Size)** | **$-2.0222$** | The physical drop in review points on a 5-star scale. |
| **95% Confidence Interval (CI)** | **$[1.9828, 2.0615]$** | We estimate the true score drop to sit between $1.98$ and $2.06$ points. |
| **Welch's $t$-statistic** | **$-100.7804$** | Extreme signal-to-noise ratio, proving a massive separation between groups. |
| **Calculated $p$-value** | **$< 0.0001$** *(Output: `0.0`)* | Virtually $0\%$ probability that this difference occurred by random chance. |

---

### Step 5: Statistical Interpretation & Business Recommendations (Slide 17, 38)

#### 1. Statistical Decision (Slide 18)
Since our $p\text{-value} < 0.0001$ is far below our significance threshold ($\alpha = 0.05$), we **reject the Null Hypothesis ($H_0$)**. The sample provides overwhelming, non-random evidence that delivery delays drastically reduce customer satisfaction scores.

#### 2. Practical & Financial Significance (Slide 25)
Beyond statistical significance, the **practical impact** on AfiLearn is critical:
* An average rating of **$4.29$** represents a premium service experience. 
* A drop to **$2.27$** represents a severe operational failure. 
* Because the 95% Confidence Interval $[1.98, 2.06]$ is extremely narrow, this negative customer response is highly predictable and stable across the entire customer base.

#### 3. Recommended Business Action (Slide 38)
* **Fund and Deploy predictive modeling (Phase 4):** Since delayed packages are statistically proven to cause severe brand damage, we recommend deploying a classification pipeline.
* **Proactive Mitigation:** When our predictive model flags an order as "high risk for delay," automated backend tasks should trigger proactively (e.g., emailing the customer ahead of time with a discount voucher) to protect brand loyalty before the package even leaves the warehouse.

---

### Operational & Ethical Limitations

Before scaling this system, we must manage three core design constraints:
1.  **Response Bias:** Review scores are voluntary. Customers are highly motivated to write reviews when they are extremely angry (late deliveries) or extremely happy, which can skew the extremes of our sample.
2.  **Causal Attribution:** While the Welch's t-test proves an association, other factors (e.g., product damage, bad communication) might occur alongside delivery delays and contribute to the score drop.
3.  **Operational Capacity Constraints:** Proactively flagging late packages will increase customer service tickets. We must run a slow, phased pilot rollout to ensure customer support teams are not overwhelmed by false alarms (Type I errors).